# Training of the student

What we do:
1) Load the training dataset
2) Run training using the huggingface client
3) Save the model


Student: Qwen2.5-1.5B-Instruct

In [1]:
import dotenv

dotenv.load_dotenv()

True

In [13]:
from core.types import *
from core.utils.huggingface_training_client import HuggingFaceTrainingClient
from core.utils.ollama_inference_client import OllamaInferenceClient
from core.utils.openai_client import OpenAIClient, ProcessingMode
from doom.preprocessing.doom_game_state_perturbator import DoomGameStatePerturbator
from doom.utils.doom_game_state import DoomGameState, MonsterType, WeaponName, AimedAtType
from sklearn.cluster import DBSCAN
from dataclasses import dataclass, asdict
from collections import Counter
from typing import Iterable
from pathlib import Path
from openai.types.responses import Response as OpenAIResponse

import os
import json
import numpy as np
import pandas as pd

In [11]:
dataset_path = Path("data/outputs/selected-data-low-reasoning-gpt5.csv")

df = pd.read_csv(dataset_path)
df.dropna(axis=0, how='all', inplace=True)

In [12]:
df

,input_id,game_state,command,command_intent,command_explicitness,command_atomicity,command_contextuality,game_actions,latency,reason_if_failed,...,action_full_correct,action_unnecessary,action_imprecise_sequentiality,action_imprecise_parameters,action_harming_sequentiality,action_harming_parameters,action_missing,action_harming,action_wrong_syntax,label
0,state-233-p2-uc0,AIMED_AT:\n type: Wall\n distance: 330.86\n ...,Walk up to that switch and use it,Approach the interactable wall and trigger its...,0.85,0.35,0.90,MOVE 0.0 330.86\nINTERACT,4.414740,NaN,...,2.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN
1,state-6603-p1-uc0,AIMED_AT:\n type: Wall\n distance: 281.96\n ...,Go press that switch ahead,Approach the interactable wall in front and ac...,0.90,0.40,0.85,SPRINT 0.0 281.96\nINTERACT,3.383618,NaN,...,2.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN
2,state-7768-p3-uc1,AIMED_AT:\n type: Wall\n distance: 736.10\n ...,Turn left and kill the enemy,Rotate toward the visible monster and eliminat...,0.80,0.40,0.90,ROTATE_TO_TARGET MONSTER_0\nFIRE_SHOTS 3,2.579986,NaN,...,2.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN
3,state-7825-p3-uc1,AIMED_AT:\n type: Wall\n distance: 675.93\n ...,Turn left and blast the closest guy,Face and kill the nearest zombieman threatenin...,0.85,0.50,0.90,ROTATE_TO_TARGET MONSTER_0\nFIRE 0.7,4.923402,NaN,...,2.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN
4,state-8097-p0-uc0,AIMED_AT:\n type: Wall\n distance: 783.61\n ...,Drop the closest zombie with the shotgun,Quickly kill the nearest low-health zombieman ...,0.85,0.40,0.90,ROTATE_TO_TARGET MONSTER_1\nFIRE_SHOTS 1,4.482084,NaN,...,2.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN
5,state-8097-p2-uc1,AIMED_AT:\n type: Wall\n distance: 783.61\n ...,Focus fire on the imp to the right,Quickly eliminate the more dangerous distant D...,0.85,0.60,0.85,ROTATE_TO_TARGET MONSTER_2\nFIRE_SHOTS 4,3.514137,NaN,...,2.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN
6,state-11021-p0-uc1,AIMED_AT:\n type: Monster\n distance: 685.57...,Center aim on that imp and fire,Adjust aim to the visible imp and shoot it once,0.85,0.50,0.85,ROTATE_TO_TARGET MONSTER_0\nFIRE 1.0,2.726541,NaN,...,2.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN
7,state-11066-p1-uc0,AIMED_AT:\n type: Wall\n distance: 319.04\n ...,Line up on that imp and fire,Aim directly at the visible DoomImp and shoot it,0.90,0.45,0.90,ROTATE_TO_TARGET MONSTER_0\nFIRE 0.5,4.505648,NaN,...,2.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN
8,state-11798-p1-uc1,AIMED_AT:\n type: Wall\n distance: 59.31\n ...,Switch to the shotgun and blast the imp,Change to the shotgun weapon and attack the ne...,0.90,0.50,0.85,SELECT Shotgun\nROTATE_TO_TARGET MONSTER_0\nFI...,4.094549,NaN,...,3.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN
9,state-18060-p0-uc0,AIMED_AT:\n type: Wall\n distance: 252.12\n ...,Turn left and blast that imp,Rotate toward the distant imp and attack it im...,0.85,0.40,0.90,ROTATE_TO_TARGET MONSTER_0\nFIRE_SHOTS 1,4.712366,NaN,...,2.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN


In [ ]:
# Extract only <inputs, labels>
# For inputs, we do not want (nor need) the full prompt. Just a couple of general pieces of information.
# That is, the inputs are SMALL PROMPT + GAME_STATE + USER COMMAND
# The labels are the df.labels

dataset = [

    for row in df.values
]

In [16]:
training_client = HuggingFaceTrainingClient(
    model="Qwen/Qwen2.5-1.5B-Instruct",
    device="cuda",
    working_dir=Path("data/huggingface"),
    use_qlora=False,
    use_flash_attention_2=False
)

🏋️  Initialized HuggingFaceTrainingClient for Qwen/Qwen2.5-1.5B-Instruct
   Device: cuda
   QLoRA: False
   Flash Attention 2: False


In [ ]:
training_client.fine_tune(

)